# XAI in the Fast Food Industry

## Table of Contents

1. Overview
2. Q&As
3. Dependencies
4. Data
5. Model Training
6. Model Evaluation
7. Model Interpretation
8. Summary
9. Exercises

## 1. Overview

The fast food industry is a behemoth, serving millions daily. But with great power comes great responsibility - and a whole lot of data. Imagine if we could use machine learning to predict customer satisfaction or identify factors leading to food waste. Wouldn't that be not only ideal, but also desirable by everyone from CEOs to the guy flipping burgers? I think it is, but keep in mind that a simple message saying "Hey, your fries are making people sad!" won't suffice. Context and a good explanation will go a long way and can provide useful information for franchisees and menu planners alike.

With this overview out of the way, let's examine some questions our fast food overlords may want answered.

## 2. Q&As

When talking to an AI that used an algorithm to tell me how to improve my fast food joint, I'd certainly ask:

1. Why are customers unsatisfied with our burgers?
2. What led to this conclusion?
3. What should we do next?

Some potential answers "I" would like to hear if my AI was telling me that my burgers are subpar:
1. Your patty-to-bun ratio is off, and this, along with other variables, contributed to low satisfaction scores. The ideal ratio is 1:1.5, yours is at 1:2.
2. The combination of oversized buns, underseasoned patties, and lukewarm serving temperature decreased the likelihood of customer satisfaction compared to other restaurants similar to yours.
3. We need to adjust your burger recipe, retrain staff on proper cooking temperatures, and conduct more regular quality checks. We'll also ask customers for more frequent feedback to track improvements.

That would make me feel a bit more positive about the situation. But there are many factors leading to fast food satisfaction, so let's dive into our greasy example. 🍔

## 3. Dependencies

Here are the packages we will be using in this notebook.

- `scikit-learn`
- `pandas`
- `joblib`
- `matplotlib`
- `alibi`
- `statsmodels`
- `mlserver`

In [ ]:
!pip install scikit-learn pandas joblib matplotlib alibi numpy rich mlserver

## 4. Data

The dataset we'll be using is a synthetic one based on fast food restaurant metrics. It contains information such as order preparation time, food temperature, customer wait time, and overall satisfaction scores.

Why it matters? Machine learning is great at finding patterns in data, and we should use these tools to enhance the fast food experience while keeping customer information safe. Many businesses fail due to customer dissatisfaction, so if there's a way to improve service while maintaining privacy, we should be flipping that digital spatula as fast as we can.

Description of variables:
- `PrepTime` - order preparation time in minutes
- `FoodTemp` - food temperature at serving (°C)
- `WaitTime` - customer wait time in minutes
- `OrderAccuracy` - accuracy of order fulfillment (0-100%)
- `StaffFriendliness` - staff friendliness rating (1-5)
- `RestaurantCleanliness` - cleanliness rating (1-5)
- `MenuVariety` - variety of menu items (1-5)
- `PriceRating` - price competitiveness (1-5)
- `LoyaltyProgram` - customer part of loyalty program (0 or 1)
- `Satisfaction` - target variable, overall satisfaction score (1-10)

Let's start by loading and evaluating our data. Hope you're hungry for some bytes!

In [ ]:
from sklearn.model_selection import train_test_split
from rich import print
import pandas as pd

# Load synthetic fast food data
df = pd.read_csv('data/fastfood/customer_satisfaction.csv')
print(df.head())

y = df['Satisfaction']
X = df.drop(['Satisfaction'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
y = df['Satisfaction']
X = df.drop(['Satisfaction'], axis=1).copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=9)

## 5. Model Training

For this section, we'll use a logistic regression because of its high interpretability and ease of use. It's like the drive-thru of machine learning models - quick, reliable, and you know what you're getting.

If you've never used logistic regression before, think of it as a classification algorithm used to predict a binary outcome (e.g., satisfied or unsatisfied, will return or won't return).

Imagine you're a fast food manager trying to predict if a customer will recommend your restaurant (a binary yes/no outcome) based on wait time, food quality, and other variables. Your process might look like this:

1. Convert the output to a probability between 0-1, representing the chance of a recommendation.

2. Use a linear model to combine input features and calculate a 'score':
   
   $score = Intercept + WaitTime * \beta_1 + FoodQuality * \beta_2$

3. Convert this score to a probability using the logistic function:
   
   $probability = \frac{1}{(1 + e^{(-score)})}$

4. If probability > 0.5, predict the customer will recommend. Otherwise, predict they won't.

Now, let's train our model. May the odds be ever in your flavor!

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set(rc={'figure.figsize':(11.7,8.27)})

Feel free to experiment with the parameters below. It's like seasoning your model - a pinch here, a dash there, until it's just right.

In [ ]:
lr_cls = LogisticRegression(random_state=0, max_iter=500, verbose=0)

In [ ]:
lr_cls.fit(X_train, y_train)

In [ ]:
lr_cls.coef_

If you don't have the path below, you can create with the following command in the termianl.

```sh
mkdir -p models/diabetes/
```

In [ ]:
model_path = 'models/diabetes/lr_cls_diabetes.pkl'

In [ ]:
joblib.dump(lr_cls, model_path)

In [ ]:
!ls models/diabetes

In [ ]:
lr_cls = joblib.load(model_path)

Let's do a quick sanity check before we move on to thoroughly evaluating our model. For this, we will 
pick a random sample from the test dataset.

In [ ]:
x = X_test.sample(1)
y = y_test[x.index[0]]
print(f"Actual satisfaction score: {y}")
print("\nCustomer attributes:")
print(x)

In [ ]:
lr_cls.classes_

In [ ]:
lr_cls.predict_proba(x)

In [ ]:
y_pred = lr_cls.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

In [ ]:
title = 'Confusion matrix for Logistic Regression'
disp = ConfusionMatrixDisplay.from_estimator(
    lr_cls, X_test, y_test, 
    display_labels=['Not Diabetic', 'Diabetic'],
    cmap=plt.cm.Blues, normalize=None
)
disp.ax_.set_title(title);

## 6. Model Evaluation

The first method we'll explore is called Kernel SHAP. It's like a food critic for your model, breaking down each ingredient's contribution to the final dish.

Here's an analogy to understand Kernel SHAP:

Imagine a burger rating model. The ingredients are patty, bun, lettuce, cheese, and special sauce. The model predicts how tasty the burger will be.

To explain an individual prediction, Kernel SHAP is like asking:

"How much did each ingredient contribute to the overall tastiness?"

It determines the SHAP value, or impact, of each feature by comparing burgers with and without that ingredient. The patty might get a high positive SHAP value because it's crucial for tastiness. Lettuce might have a low or negative SHAP value if it doesn't add much to the flavor. By summing the SHAP values for all features, Kernel SHAP explains the total predicted tastiness.

Like this, Kernel SHAP attributes the prediction of any complex model to each input feature. The analogy helps convey how it quantifies each feature's contribution, like ingredients in a recipe. This makes model behavior more interpretable than a secret sauce.

Let's get cooking with KernelShap!

In [ ]:
from alibi.explainers import KernelShap

In [ ]:
explainer = KernelShap(lr_cls.predict_proba, task='classification')
explainer

Explainers in Alibi work in the same fashion as estimators in sklearn, that is, they follow the 
`.fit()` and `.predict()` way of doing things so if you are familiar with sklearn, this step will 
feel familiar to you.

In [ ]:
explainer.fit(X_train)

Once we finish creating an explainer, the object we get back gives us a lot of useful information like the one above.

Note that, running an explainer in a large batch of data can be quite compute intensive (depending on the 
explainer of course), so it is good practice to save your models once your code finishes creating them. Let's 
save ours, load it and test it again.

In [ ]:
explainer_path = 'models/diabetes/lr_cls_explainer.pkl'

In [ ]:
joblib.dump(explainer, explainer_path)

In [ ]:
explainer = joblib.load(explainer_path)

In [ ]:
x = X_test.sample(1)
y = y_test[x.index].iloc[0]
print(y)
x

In [ ]:
features = X_train.columns.to_list()
features

As you might have noticed in the metadata returned when we trained our model, KernelShap is both local and global. It's like being able to explain why a specific customer loved their burger, and also why customers in general love your burgers. Let's try it on our random sample from above.

In [ ]:
result = explainer.explain(x)

In [ ]:
result.shap_values[0]

In [ ]:
explainer.predictor(x)

What we're interested in is the `shap_values` returned by our explainer. Let's see what these look like.

In [ ]:
result.shap_values

In [ ]:
def plot_importance(feat_imp, feat_names, class_idx):
    df = pd.DataFrame(data=feat_imp, columns=feat_names).sort_values(by=0, axis='columns')
    feat_imp, feat_names = df.values[0], df.columns
    fig, ax = plt.subplots(figsize=(10, 5))
    y_pos = np.arange(len(feat_imp))
    ax.barh(y_pos, feat_imp)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feat_names, fontsize=15)
    ax.invert_yaxis()
    ax.set_xlabel(f'Feature effects for class {class_idx}', fontsize=15)
    return ax, fig

In [ ]:
import numpy as np

In [ ]:
plot_importance(result.shap_values[1], features, 'High Satisfaction');

In [ ]:
import shap

In [ ]:
result = explainer.explain(X_train[:100])

In [ ]:
shap.summary_plot(result.shap_values[0], X_train[:100], features);

A positive SHAP value means the feature pushed the output higher. Negative means it pushed the output lower.

It is important to note that, if we train the explainer on a large amount of data (with some compute expenses), the 
explainer would have learned enough about the model globally to locally explain the interactions for new cases.

## 7. Model Interpretation

While Kernel Shap is considered a black-box method (like the secret recipe for your special sauce), because we chose logistic regression as our model, we can also interrogate each of the coefficients and interpret the results further.

To do this, we'll use statsmodels to fit a model again because it serves up a very nice summary table. It's like getting the nutritional information for your model.

In [ ]:
# !pip install 'alibi[shap]'
!pip install 'alibi[ray]'

In [ ]:
import statsmodels.api as sm

In [ ]:
log_model = sm.Logit(y_train, sm.add_constant(X_train))
log_result = log_model.fit()

In [ ]:
print(log_result.summary2())

In the table above we can examine not only the coefficients of each parameter, but also the standard deviation and 
AIC and BIC values of our model.

Because the coefficients are the logarithms of the odds (i.e. the probability of a positive case over 
the probability of a negative case), we can convert them back into exponentials to get a better sense of 
what each value means.

In [ ]:
np.exp(log_result.params).sort_values(ascending=False)

What do the odds mean for a satisfied customer? It means that the odds of high satisfaction increase by a factor of X for each additional unit of Y, provided every other feature stays unchanged. It's like saying adding an extra pickle increases the chances of a 5-star review by 20%, assuming everything else stays the same.

We need more context for this, and that can be achieved with the standard deviation. It's like understanding how much each ingredient varies across all your burgers.

In [ ]:
coefs = log_result.params.drop(labels=['const'])
stdv = np.std(X_train, 0)
abs(coefs * stdv).sort_values(ascending=False)

The preceding table can be interpreted as an approximation of risk factors from high to low 
according to the model. It is also a model-specific feature importance method, and a global one 
at that (as it was gather from a group of samples). It tells us how far away from the mean each of 
these values are.

1. Explainable AI (XAI) in the fast food industry can help understand customer satisfaction drivers, optimize menu items, and improve overall service quality.

2. Kernel SHAP helps break down how each factor (like wait time, food temperature) influences customer satisfaction, much like understanding how each ingredient contributes to a burger's taste.

3. Logistic regression can be used to predict binary outcomes like whether a customer will return or recommend the restaurant.

4. XAI enhances transparency and trust, crucial in the fast food industry where customer experience can make or break a business.

5. Machine Learning can assist in predicting customer behavior, optimizing operations, and personalizing marketing strategies in the fast food sector.

1. Explainable AI (XAI) is a set of techniques that make AI models more transparent, helping fast food managers understand why customers love or hate their burgers.

2. Kernel SHAP is like a food critic for your AI, breaking down how each 'ingredient' (feature) contributes to the final 'taste' (prediction).

3. Logistic regression is the drive-thru of machine learning models - quick, reliable, and predicts binary outcomes like "will this customer come back for seconds?"

4. XAI in fast food can enhance customer satisfaction, optimize menus, and make operations smoother than a well-blended milkshake.

5. Machine Learning can help predict which new menu item will be a hit, or which locations might need extra staff during rush hour, potentially turning your fast food joint into the next big cheese.

## Bonus: Serving our Models

To serve models and explainers together, you can run a server with both models using `mlserver`. 
To do so, run the following command.

```sh
python servers/diabetes/cls_diabetes_service.py
```

You can test that your server is working with the following commands.

In [ ]:
from mlserver.codecs import NumpyCodec
import requests

In [ ]:
x.values, y

In [ ]:
inf_request = {
    'inputs': [
        NumpyCodec.encode_input(name='payload', payload=x.values).dict()
    ]
}
print(inf_request)

Change the name from **classifier** to **explainer** and back to change the endpoint you 
are hitting.

In [ ]:
model = 'diabetes_classifier'
endpoint = f"http://0.0.0.0:8080/v2/models/{model}/infer"
r = requests.post(endpoint, json=inf_request)
r.json()

## 10. Exercises

### Build an Explainer on Fast Food Customer Retention

- Load a dataset on fast food customer behavior. If you can't find one, whip up some synthetic data faster than you can say "Would you like fries with that?"
- Train a logistic regression model to predict if a customer will return within 30 days.
- Create a Kernel Shap explainer using Alibi, or use another method like AnchorTabular or CounterFactual. It's like choosing between different sauce options - they're all good, but some might suit your taste better.
- Write a short narrative describing the result. Make it snappier than fast food advertising.
- If you're feeling extra hungry for knowledge, compare explanations from different models. Which features do they highlight? How do they differ? It's like comparing menu items - they might all be burgers, but the toppings can make a world of difference.